# Capa 3: Analítica y Agregación de Negocio (Gold)
**Objetivo:** Consolidar métricas ejecutivas (KPIs), calcular estadísticos de tendencia central (Edad, CD4, Oportunidad TAR) y estructurar las agregaciones geográficas por municipio y subregión para el consumo directo de la aplicación en Streamlit.

Carga del Dataset Silver

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Definición portátil de rutas
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_SILVER = BASE_DIR / "data" / "silver"
DIR_GOLD = BASE_DIR / "data" / "gold"

# Crear directorio Gold si no existe
DIR_GOLD.mkdir(parents=True, exist_ok=True)

# Cargar dataset depurado en Silver
ruta_silver = DIR_SILVER / "cohorte_vih_silver.parquet"
if not ruta_silver.exists():
    ruta_silver = DIR_SILVER / "cohorte_vih_silver.csv"
    df_silver = pd.read_csv(ruta_silver)
else:
    df_silver = pd.read_parquet(ruta_silver)

# Validación Celda 2
print(" Dataset Silver cargado en la Capa Gold:")
print(
    f"   Total Registros: {len(df_silver):,} | Total Columnas: {len(df_silver.columns)}"
)

 Dataset Silver cargado en la Capa Gold:
   Total Registros: 850 | Total Columnas: 124


Cálculo de Medidas de Tendencia Central

In [2]:
# Variables cuantitativas clave para el análisis epidemiológico
col_cd4 = "75.1 Resultado del último conteo de linfocitos TCD4"
col_tar_dias = "Días oportunidad inicio TAR"

# Cálculo de estadísticas descriptivas (Media, Mediana, Desviación Estándar, Mín, Máx)
tendencia_central = pd.DataFrame(
    {
        "Variable_Analizada": [
            "Edad (Años)",
            "Conteo Linfocitos CD4 (cel/mm³)",
            "Días Oportunidad Inicio TAR",
        ],
        "Promedio_Media": [
            round(df_silver["Edad"].mean(), 2),
            round(df_silver[col_cd4].mean(), 2),
            round(df_silver[col_tar_dias].mean(), 2),
        ],
        "Mediana": [
            df_silver["Edad"].median(),
            df_silver[col_cd4].median(),
            df_silver[col_tar_dias].median(),
        ],
        "Desviacion_Estandar": [
            round(df_silver["Edad"].std(), 2),
            round(df_silver[col_cd4].std(), 2),
            round(df_silver[col_tar_dias].std(), 2),
        ],
        "Valor_Minimo": [
            df_silver["Edad"].min(),
            df_silver[col_cd4].min(),
            df_silver[col_tar_dias].min(),
        ],
        "Valor_Maximo": [
            df_silver["Edad"].max(),
            df_silver[col_cd4].max(),
            df_silver[col_tar_dias].max(),
        ],
    }
)

# Validación Celda 3
print(" Medidas de Tendencia Central y Dispersión (Cohorte VIH Antioquia):")
display(tendencia_central)

 Medidas de Tendencia Central y Dispersión (Cohorte VIH Antioquia):


,Variable_Analizada,Promedio_Media,Mediana,Desviacion_Estandar,Valor_Minimo,Valor_Maximo
0,Edad (Años),46.70,46.0,17.02,18,75
1,Conteo Linfocitos CD4 (cel/mm³),700.89,701.0,283.22,200,1198
2,Días Oportunidad Inicio TAR,24.07,24.5,12.77,3,45


Agregación Ejecutiva de KPIs DULCINEA

In [3]:
# Totales globales de la cohorte
total_pacientes = len(df_silver)
abandonos = (
    df_silver["Estado_Seguimiento_VIH"] == "Abandono / Sin TAR"
).sum()
controlados = (
    df_silver["Estado_Seguimiento_VIH"] == "En TAR - Virológicamente Controlado"
).sum()
riesgo_falla = (
    df_silver["Estado_Seguimiento_VIH"] == "En TAR - Con Carga Viral Detectable"
).sum()

tasa_abandono = round((abandonos / total_pacientes) * 100, 2)
tasa_control = round((controlados / total_pacientes) * 100, 2)
tasa_falla = round((riesgo_falla / total_pacientes) * 100, 2)

kpi_ejecutivos = pd.DataFrame(
    {
        "Indicador_DULCINEA": [
            "Total Cohorte Evaluada",
            "Pacientes en Abandono / Sin TAR",
            "Pacientes en TAR - Virológicamente Controlados",
            "Pacientes en TAR - Con Falla Viral Detectable",
            "Tasa Global de Abandono (%)",
            "Tasa de Control Virológico (%)",
            "Tasa de Riesgo por Falla Viral (%)",
        ],
        "Valor": [
            f"{total_pacientes:,}",
            f"{abandonos:,}",
            f"{controlados:,}",
            f"{riesgo_falla:,}",
            f"{tasa_abandono}%",
            f"{tasa_control}%",
            f"{tasa_falla}%",
        ],
    }
)

# Validación Celda 4
print(" Tablero Ejecutivo de Indicadores (DULCINEA):")
display(kpi_ejecutivos)

 Tablero Ejecutivo de Indicadores (DULCINEA):


,Indicador_DULCINEA,Valor
0,Total Cohorte Evaluada,850
1,Pacientes en Abandono / Sin TAR,217
2,Pacientes en TAR - Virológicamente Controlados,451
3,Pacientes en TAR - Con Falla Viral Detectable,182
4,Tasa Global de Abandono (%),25.53%
5,Tasa de Control Virológico (%),53.06%
6,Tasa de Riesgo por Falla Viral (%),21.41%


Agregación Geográfica por Municipio y Subregión

In [4]:
# Agrupación por Subregión, Municipio de Residencia y Estado de Seguimiento
kpi_territorial = (
    df_silver.groupby(
        ["Subregion", "Municipio residencia", "Estado_Seguimiento_VIH"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Garantizar columnas del pivote
for col in [
    "Abandono / Sin TAR",
    "En TAR - Con Carga Viral Detectable",
    "En TAR - Virológicamente Controlado",
]:
    if col not in kpi_territorial.columns:
        kpi_territorial[col] = 0

# Calcular Totales y Tasas Locales
kpi_territorial["Total_Pacientes"] = (
    kpi_territorial["Abandono / Sin TAR"]
    + kpi_territorial["En TAR - Con Carga Viral Detectable"]
    + kpi_territorial["En TAR - Virológicamente Controlado"]
)

kpi_territorial["Tasa_Abandono_%"] = (
    (kpi_territorial["Abandono / Sin TAR"] / kpi_territorial["Total_Pacientes"])
    * 100
).round(2)

kpi_territorial = kpi_territorial.sort_values(
    by="Abandono / Sin TAR", ascending=False
)

# Validación Celda 5
print(" Matriz Geográfica de Abandono por Municipio (Top 5 con mayor desafección):")
display(kpi_territorial.head(5))

 Matriz Geográfica de Abandono por Municipio (Top 5 con mayor desafección):


Estado_Seguimiento_VIH,Subregion,Municipio residencia,Abandono / Sin TAR,En TAR - Con Carga Viral Detectable,En TAR - Virológicamente Controlado,Total_Pacientes,Tasa_Abandono_%
1,Oriente,Marinilla,29,17,42,88,32.95
5,Valle de Aburrá,Girardota,28,25,41,94,29.79
8,Valle de Aburrá,Medellín (Sabaneta),25,14,40,79,31.65
9,Valle de Aburrá,Sabaneta,24,12,41,77,31.17
4,Valle de Aburrá,Envigado,23,21,47,91,25.27


### Módulo de Predicción y Estimación de Riesgo de Abandono
**Objetivo:** Implementar un algoritmo de aprendizaje supervisado (Regresión Logística) para predecir la probabilidad de deserción del tratamiento o falla virológica y estratificar a los pacientes en tres niveles de riesgo preventivo (Riesgo Bajo, Medio y Alto).

In [5]:
# !pip install scikit-learn

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# 1. Rutas de carpetas
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_SILVER = BASE_DIR / "data" / "silver"
DIR_GOLD = BASE_DIR / "data" / "gold"

# Cargar dataset de la Capa Silver
df_silver = pd.read_parquet(DIR_SILVER / "cohorte_vih_silver.parquet")

# 2. Definir Target de Riesgo Alto (Abandono O Falla Viral Detectable)
df_silver["falla_viral"] = df_silver[
    "76.1 Resultado de la última Carga viral para VIH"
].apply(
    lambda x: 0
    if "indetectable" in str(x).lower() or str(x) in ["<50", "0"]
    else 1
)

df_silver["target_riesgo"] = (
    (df_silver["es_abandono"] == 1) | (df_silver["falla_viral"] == 1)
).astype(int)

# CORRECCIÓN: sin saber si el paciente recibe TAR, el modelo confunde
# "pocas semanas en TAR porque acaba de iniciar (va bien)" con
# "pocas semanas en TAR porque abandonó (riesgo alto)". Se agrega la bandera.
df_silver["recibe_tar_bin"] = (df_silver["77. Recibe TAR"].astype(str).str.strip() == "Sí").astype(int)

# 3. Entrenar Modelo de Regresión Logística (Estimación / Predicción)
variables_predictores = [
    "Edad",
    "Semanas en TAR",
    "75.1 Resultado del último conteo de linfocitos TCD4",
    "Días oportunidad inicio TAR",
    "recibe_tar_bin",
]
X = df_silver[variables_predictores].fillna(0)
y = df_silver["target_riesgo"]

modelo_estimador = LogisticRegression(max_iter=1000, random_state=42)
modelo_estimador.fit(X, y)

# 4. Calcular Probabilidades e Infectar Niveles de Riesgo Estimados
df_silver["Probabilidad_Abandono_%"] = (
    modelo_estimador.predict_proba(X)[:, 1] * 100
).round(2)

df_silver["Nivel_Riesgo_Estimado"] = pd.cut(
    df_silver["Probabilidad_Abandono_%"],
    bins=[-1, 30, 60, 100],
    labels=["Riesgo Bajo", "Riesgo Medio", "Riesgo Alto"],
)

# 5. Exportar Tabla de Predicción a Capa Gold
ruta_predicciones = DIR_GOLD / "prediccion_riesgo_abandono.csv"
df_silver[
    [
        "Numero de Identificacion",
        "Nombre_Completo",
        "Municipio residencia",
        "Subregion",
        "Estado_Seguimiento_VIH",
        "Probabilidad_Abandono_%",
        "Nivel_Riesgo_Estimado",
    ]
].to_csv(ruta_predicciones, index=False, encoding="utf-8-sig")

# Validación Celda 8
print(" MODELO DE PREDICCIÓN / ESTIMACIÓN PROCESADO:")
print(df_silver["Nivel_Riesgo_Estimado"].value_counts())

 MODELO DE PREDICCIÓN / ESTIMACIÓN PROCESADO:
Nivel_Riesgo_Estimado
Riesgo Bajo     420
Riesgo Alto     217
Riesgo Medio    213
Name: count, dtype: int64


Exportación de Tablas Agregadas a la Capa Gold

In [7]:
# Rutas de salida en data/gold/
ruta_tendencia = DIR_GOLD / "tendencia_central_vih.csv"
ruta_kpis = DIR_GOLD / "kpi_ejecutivos_dulcinea.csv"
ruta_territorial = DIR_GOLD / "kpi_territorial_abandono.csv"

# Guardar tablas CSV de consumo directo
tendencia_central.to_csv(ruta_tendencia, index=False, encoding="utf-8-sig")
kpi_ejecutivos.to_csv(ruta_kpis, index=False, encoding="utf-8-sig")
kpi_territorial.to_csv(ruta_territorial, index=False, encoding="utf-8-sig")

# Validación Celda 6
assert ruta_tendencia.exists(), "Error al guardar tendencia_central_vih.csv"
assert ruta_kpis.exists(), "Error al guardar kpi_ejecutivos_dulcinea.csv"
assert ruta_territorial.exists(), "Error al guardar kpi_territorial_abandono.csv"

print(" CAPA GOLD PROCESADA Y EXPORTADA CON ÉXITO")
print(f"   - Tendencia Central: {ruta_tendencia.name}")
print(f"   - KPIs Ejecutivos:   {ruta_kpis.name}")
print(f"   - Agregación Geográfica: {ruta_territorial.name}")

 CAPA GOLD PROCESADA Y EXPORTADA CON ÉXITO
   - Tendencia Central: tendencia_central_vih.csv
   - KPIs Ejecutivos:   kpi_ejecutivos_dulcinea.csv
   - Agregación Geográfica: kpi_territorial_abandono.csv
